# Sesi 2 — 05. Uji Ingatan: Kapan Gerbang LSTM & GRU Penting?

**Workshop Deep Learning untuk Mahasiswa Teknik Telekomunikasi — ITERA**

Di notebook 03, RNN biasa ternyata **sama baiknya** dengan GRU dan LSTM. Alasannya: meramal trafik 1 jam ke depan hanya butuh beberapa jam terakhir. Tugas itu **tidak menuntut ingatan panjang**.

Di sini kita buat tugas yang **memaksa** model mengingat sesuatu dari jauh di masa lalu: *adding problem* (Hochreiter & Schmidhuber, 1997).

```
langkah  :   1     2     3     4   …   61    62   …   100
nilai    :  0.42  0.17  0.93  0.55 …  0.08  0.71 …  0.36
penanda  :   0     1     0     0   …   0     1   …   0
                   ↑                         ↑
target   : 0.17 + 0.71 = 0.88   (jumlah dua nilai yang ditandai)
```

- Penanda pertama selalu di **10% awal** urutan, penanda kedua di **paruh akhir**.
- Model membaca satu langkah per waktu, sehingga nilai bertanda pertama harus **diingat** sampai akhir urutan.
- Kita uji panjang urutan **T = 10, 50, 100, 200**.

| Bagian | Isi |
|---|---|
| 1 | Membuat data *adding problem* |
| 2 | Tiga model dengan jumlah parameter setara |
| 3 | Training |
| 4 | Hasil: error menurut panjang urutan |
| 5 | Kurva belajar |
| 6 | Ke mana model "melihat"? |

In [ ]:
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Palet warna ITERA (sama dengan slide)
NAVY, GOLD, RED, TEAL, GREEN, GRAY = "#0B1F3A", "#D4A017", "#C8102E", "#1E8C8C", "#2E9E6B", "#5A5A5A"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25})

WARNA_MODEL = {"RNN": RED, "LSTM": NAVY, "GRU": TEAL}

# ---- Pengaturan eksperimen ----
CEPAT = False                 # True = 1 seed & langkah lebih sedikit (untuk laptop tanpa GPU)
PANJANG = [10, 50, 100, 200]  # panjang urutan T
SEEDS = [0] if CEPAT else [0, 1]
LANGKAH = 1500 if CEPAT else 4000   # jumlah update bobot
BATCH_SIZE = 128
LR = 1e-3
CEK_SETIAP = 100              # evaluasi val setiap 100 langkah

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HASIL_DIR = Path("hasil") / "ingatan"
HASIL_DIR.mkdir(parents=True, exist_ok=True)
print("PyTorch", torch.__version__, "| device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

---
## 1. Data *Adding Problem*

Setiap langkah punya 2 fitur: **nilai** acak 0–1 dan **penanda** (0 atau 1). Data dibuat acak setiap kali, sehingga model tidak bisa menghafal.

**Tebakan paling malas:** selalu menjawab 1.0 (rata-rata jumlah dua angka acak 0–1). Error MSE-nya ± **0.167**. Model yang tidak lebih baik dari ini berarti **tidak mengingat apa pun**.

In [ ]:
def buat_data(n, T, g):
    """n urutan sepanjang T. Fitur: [nilai, penanda]. Target: jumlah dua nilai bertanda."""
    nilai = torch.rand(n, T, generator=g)
    pos1 = torch.randint(0, max(1, T // 10), (n,), generator=g)        # 10% awal
    pos2 = torch.randint(T // 2, T, (n,), generator=g)                 # paruh akhir
    penanda = torch.zeros(n, T)
    baris = torch.arange(n)
    penanda[baris, pos1] = 1.0
    penanda[baris, pos2] = 1.0
    X = torch.stack([nilai, penanda], dim=2)                           # [n, T, 2]
    y = (nilai[baris, pos1] + nilai[baris, pos2]).unsqueeze(1)          # [n, 1]
    return X, y

MSE_MALAS = 1 / 6       # varians jumlah dua angka acak 0–1

# Contoh satu urutan T = 50
X, y = buat_data(1, 50, torch.Generator().manual_seed(7))
fig, ax = plt.subplots(figsize=(12, 2.8))
langkah = np.arange(1, 51)
ax.bar(langkah, X[0, :, 0], color=GRAY, alpha=0.5, label="nilai")
tanda = X[0, :, 1].bool().numpy()
ax.bar(langkah[tanda], X[0, tanda, 0], color=RED, label="nilai bertanda")
for l, v in zip(langkah[tanda], X[0, tanda, 0]):
    ax.text(l, v + 0.04, f"{v:.2f}", ha="center", color=RED, fontweight="bold")
ax.set_xlabel("langkah waktu"); ax.set_ylim(0, 1.2); ax.legend(loc="upper right")
ax.set_title(f"Contoh urutan T = 50 — target = {y.item():.2f}")
plt.tight_layout()
plt.savefig(HASIL_DIR / "contoh_urutan.png", dpi=200, bbox_inches="tight")
plt.show()

---
## 2. Tiga Model, Parameter Setara

Seperti di notebook 03, ukuran *hidden* diatur agar jumlah parameter ketiganya hampir sama (± 17 ribu). Jadi kalau RNN gagal, penyebabnya **bukan** karena modelnya lebih kecil.

ANN tidak diikutkan: ANN melihat seluruh urutan sekaligus sehingga tidak punya "masalah ingatan", tetapi ukuran inputnya harus tetap dan parameternya tumbuh mengikuti T.

In [ ]:
class Rekuren(nn.Module):
    def __init__(self, jenis, hidden):
        super().__init__()
        sel = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[jenis]
        self.rnn = sel(2, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1])

HIDDEN = {"RNN": 128, "LSTM": 64, "GRU": 74}
MODELS = ["RNN", "LSTM", "GRU"]
pd.Series({m: sum(p.numel() for p in Rekuren(m, HIDDEN[m]).parameters()) for m in MODELS}, name="parameter")

---
## 3. Training

Setiap langkah memakai *batch* baru yang dibuat acak. *Gradient clipping* dipakai agar training RNN tidak meledak.

In [ ]:
def latih(nama, T, seed):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(1000 + seed)
    Xva, yva = buat_data(1000, T, torch.Generator().manual_seed(1))
    Xva, yva = Xva.to(DEVICE), yva.to(DEVICE)

    model = Rekuren(nama, HIDDEN[nama]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()
    kurva = []
    for step in range(1, LANGKAH + 1):
        X, y = buat_data(BATCH_SIZE, T, g)
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(X.to(DEVICE)), y.to(DEVICE))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if step % CEK_SETIAP == 0:
            model.eval()
            with torch.no_grad():
                kurva.append((step, loss_fn(model(Xva), yva).item()))
    return model, np.array(kurva)


def uji(model, T):
    X, y = buat_data(2000, T, torch.Generator().manual_seed(2))
    model.eval()
    with torch.no_grad():
        pred = model(X.to(DEVICE)).cpu()
    mse = torch.mean((pred - y) ** 2).item()
    tepat = torch.mean((torch.abs(pred - y) < 0.1).float()).item() * 100   # % jawaban meleset < 0.1
    return mse, tepat


hasil, kurva_semua, model_simpan = [], {}, {}
for T in PANJANG:
    for nama in MODELS:
        for seed in SEEDS:
            t0 = time.time()
            model, kurva = latih(nama, T, seed)
            mse, tepat = uji(model, T)
            kurva_semua[(nama, T, seed)] = kurva
            if seed == SEEDS[0]:
                model_simpan[(nama, T)] = model
            hasil.append({"T": T, "model": nama, "seed": seed, "MSE": mse, "tepat_%": tepat, "detik": time.time() - t0})
            print(f"T={T:3d} {nama:4s} seed={seed}  MSE={mse:.4f}  tepat(±0.1)={tepat:5.1f}%  {time.time() - t0:5.1f} s")

hasil = pd.DataFrame(hasil)
hasil.to_csv(HASIL_DIR / "hasil_ingatan.csv", index=False)

---
## 4. Hasil: Error Menurut Panjang Urutan

In [ ]:
ringkas = hasil.groupby(["T", "model"]).agg(MSE=("MSE", "mean"), MSE_min=("MSE", "min"), MSE_max=("MSE", "max"),
                                            tepat=("tepat_%", "mean")).reset_index()
print(ringkas.pivot(index="model", columns="T", values="tepat").round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 3.9))
for nama in MODELS:
    r = ringkas[ringkas.model == nama]
    axes[0].plot(r["T"], r.MSE, color=WARNA_MODEL[nama], lw=2.2, marker="o", label=nama)
    axes[0].fill_between(r["T"], r.MSE_min, r.MSE_max, color=WARNA_MODEL[nama], alpha=0.15)
    axes[1].plot(r["T"], r.tepat, color=WARNA_MODEL[nama], lw=2.2, marker="o", label=nama)
axes[0].axhline(MSE_MALAS, color=GRAY, ls="--", lw=1.5, label="tebak 1.0 terus (0.167)")
axes[0].set_yscale("log"); axes[0].set_ylabel("MSE test (log)"); axes[0].set_title("Error (lebih kecil = lebih baik)")
axes[1].set_ylabel("% jawaban meleset < 0.1"); axes[1].set_ylim(0, 105); axes[1].set_title("Ketepatan")
for ax in axes:
    ax.set_xlabel("panjang urutan T"); ax.set_xticks(PANJANG); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(HASIL_DIR / "error_vs_panjang.png", dpi=200, bbox_inches="tight")
plt.show()

---
## 5. Kurva Belajar (T = 100)

In [ ]:
T_PLOT = 100
fig, ax = plt.subplots(figsize=(10, 3.6))
for nama in MODELS:
    for seed in SEEDS:
        k = kurva_semua[(nama, T_PLOT, seed)]
        ax.plot(k[:, 0], k[:, 1], color=WARNA_MODEL[nama], lw=1.8, alpha=1 if seed == SEEDS[0] else 0.45,
                label=nama if seed == SEEDS[0] else None)
ax.axhline(MSE_MALAS, color=GRAY, ls="--", lw=1.5, label="tebak 1.0 terus")
ax.set_yscale("log"); ax.set_xlabel("langkah training"); ax.set_ylabel("MSE val (log)")
ax.set_title(f"Kurva belajar, T = {T_PLOT}")
ax.legend()
plt.tight_layout()
plt.savefig(HASIL_DIR / "kurva_belajar_ingatan.png", dpi=200, bbox_inches="tight")
plt.show()

---
## 6. Ke Mana Model "Melihat"?

Sama seperti notebook 03, kita ukur $\left|\partial \hat{y} / \partial x_t\right|$: seberapa besar pengaruh setiap langkah input terhadap jawaban. Model yang benar-benar mengingat harus menunjukkan **lonjakan pada kedua langkah bertanda**, termasuk yang jauh di awal.

In [ ]:
T_PLOT = 100
X, y = buat_data(1, T_PLOT, torch.Generator().manual_seed(11))
tanda = np.where(X[0, :, 1].numpy() == 1)[0]

fig, axes = plt.subplots(len(MODELS), 1, figsize=(12, 6.2), sharex=True)
for ax, nama in zip(axes, MODELS):
    model = model_simpan[(nama, T_PLOT)]
    model.train()                                   # cuDNN RNN butuh mode train untuk backward
    x = X.clone().to(DEVICE).requires_grad_(True)
    pred = model(x)
    pred.sum().backward()
    g = x.grad[0].abs().sum(dim=1).cpu().numpy()
    ax.bar(np.arange(1, T_PLOT + 1), g / g.max(), color=WARNA_MODEL[nama], width=0.9)
    for p in tanda:
        ax.axvline(p + 1, color=GOLD, lw=2, ls="--")
    ax.set_ylabel(nama, fontweight="bold", color=WARNA_MODEL[nama])
    ax.text(0.99, 0.8, f"jawab {pred.item():.2f}  (benar {y.item():.2f})", transform=ax.transAxes, ha="right", fontsize=10)
axes[-1].set_xlabel("langkah waktu (garis emas = langkah bertanda)")
axes[0].set_title(f"Pengaruh tiap langkah input terhadap jawaban, T = {T_PLOT}")
plt.tight_layout()
plt.savefig(HASIL_DIR / "pengaruh_ingatan.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
ringkasan = {
    "device": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
    "torch": torch.__version__, "seeds": SEEDS, "langkah": LANGKAH, "hidden": HIDDEN,
    "ringkas": ringkas.round(4).to_dict(orient="records"),
    "per_seed": hasil.round(4).to_dict(orient="records"),
}
(HASIL_DIR / "ringkasan.json").write_text(json.dumps(ringkasan, indent=2))
print("Tersimpan di", HASIL_DIR)

---
## Kesimpulan

- Pada urutan **pendek**, ketiga model mampu. Pada urutan **panjang**, RNN biasa kesulitan membawa informasi dari langkah awal sampai akhir (*vanishing gradient*).
- **Gerbang** pada LSTM (*forget/input/output*) dan GRU (*update/reset*) membuat "jalan tol" untuk informasi: nilai penting bisa disimpan dan dibawa jauh tanpa terkikis.
- Pelajaran dari notebook 03 & 05: **pilih model sesuai kebutuhan tugas**. Bila tugas hanya butuh ingatan pendek (trafik 1 jam ke depan), RNN sederhana cukup. Bila butuh ingatan panjang, LSTM/GRU unggul.

## Latihan

1. Tambah `PANJANG` hingga 400. Apakah LSTM/GRU masih mampu?
2. Gandakan `LANGKAH`. Apakah RNN akhirnya bisa belajar T = 100?
3. Hapus `clip_grad_norm_`. Apa yang terjadi pada RNN?